In [ ]:
# Cell 1 — env check
import sys
print('python:', sys.version.split()[0])
import mosaic
print('mosaic package loaded')


In [ ]:
# Cell 2 — config (fill paths below)
from pathlib import Path
from mosaic.config import MosaicConfig

cfg = MosaicConfig(
    tile_source_dir=Path('/path/to/your/photos'),   # ← 填这里
    target_image=Path('/path/to/target.jpg'),        # ← 填这里
    grid=(120, 68),
    tile_px=16,
    candidate_k=50,
    lambda_reuse=0.3,
    mu_neighbor=0.2,
    tau_tone=0.5,
    verbose=True,
)
cfg.validate()
print('config ok')


In [ ]:
# Cell 3 — build / load tile pool (cached)
from mosaic.tiles import load_or_build

tile_pool = load_or_build(cfg.tile_source_dir, cfg.tile_px, cfg.cache_dir)
print(f'{len(tile_pool):,} tiles ready')


In [ ]:
# Cell 4 — split target into LAB cell grid
from mosaic.match import split_target

target_cells = split_target(cfg.target_image, cfg.grid)
print('target cells shape:', target_cells.shape)


In [ ]:
# Cell 5 — match each cell to a tile (verbose prints the thinking)
from mosaic.match import match_all_tiles

assignment, use_count = match_all_tiles(target_cells, tile_pool, cfg)
print(f'done. used {sum(1 for v in use_count.values() if v > 0)} / {len(tile_pool)} tiles.')


In [ ]:
# Cell 6 — render + save PNG
from mosaic.render import render_mosaic

mosaic_img = render_mosaic(assignment, tile_pool, target_cells, cfg)
out_png = cfg.output_dir / 'mosaic.png'
mosaic_img.save(out_png)
print('saved:', out_png)
mosaic_img


In [ ]:
# Cell 7 — self-deprecating report
from mosaic.report import generate_text_report, plot_usage_histogram, build_cold_wall

print(generate_text_report(use_count, tile_pool, total_cells=cfg.grid[0]*cfg.grid[1]))
plot_usage_histogram(use_count)
build_cold_wall(tile_pool, use_count)


In [ ]:
# Cell 8 — DeepZoom HTML export
from mosaic.deepzoom import export_deepzoom

html = export_deepzoom(mosaic_img, cfg.output_dir)
print(f'open: file://{html}')
